In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold, GroupKFold, StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load the dataset
df = pd.read_csv('diabetes (1).csv')
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Synthesize a 'Group' parameter to demonstrate Group K-Folds (Methods 8 & 9)
np.random.seed(42)
groups = np.random.randint(0, 100, size=len(df)) # 100 fake patient IDs

# Global list to accumulate metrics across all random seeds
all_records = []

# Helper function to compute and track metrics per seed
def record_metrics(technique_name, y_true, y_pred, seed_val):
    all_records.append({
        'Seed': seed_val,
        'Technique Splitting Design': technique_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-Score': f1_score(y_true, y_pred, zero_division=0)
    })


print("--- Processing Splitting Designs on Train/Test Split Pool (Seeds 2 to 100) ---\n")

# Loop through random seed values from 2 to 100
for seed in range(2, 101):

    # Generate the pristine 80% training pool and 20% test set from the original dataset
    X_train_pool, X_test_pool, y_train_pool, y_test_pool = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )

    # Align group constraints to match the active training pool
    groups_train = groups[X_train_pool.index]

    # -----------------------------------------------------------------------------
    # 1. Random Hold-Out
    # -----------------------------------------------------------------------------
    np.random.seed(seed)
    train_idx = np.random.choice(X_train_pool.index, size=int(0.8 * len(X_train_pool)), replace=False)
    X_train = X_train_pool.loc[train_idx]
    y_train = y_train_pool.loc[train_idx]

    # Initialize and fit a clean Decision Tree variant inside this technique block
    dt_model_1 = DecisionTreeClassifier(random_state=seed)
    dt_model_1.fit(X_train, y_train)
    record_metrics("1. Random Hold-Out", y_test_pool, dt_model_1.predict(X_test_pool), seed)


    # -----------------------------------------------------------------------------
    # 2. Shuffled Hold-Out
    # -----------------------------------------------------------------------------
    train_pool_df = X_train_pool.copy()
    train_pool_df['Outcome'] = y_train_pool
    df_shuffled = train_pool_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    X_shuf = df_shuffled.drop('Outcome', axis=1)
    y_shuf = df_shuffled['Outcome']
    split_point = int(0.8 * len(df_shuffled))

    X_train = X_shuf.iloc[:split_point]
    y_train = y_shuf.iloc[:split_point]

    # Initialize and fit a clean Decision Tree variant inside this technique block
    dt_model_2 = DecisionTreeClassifier(random_state=seed)
    dt_model_2.fit(X_train, y_train)
    record_metrics("2. Shuffled Hold-Out", y_test_pool, dt_model_2.predict(X_test_pool), seed)


    # -----------------------------------------------------------------------------
    # 3. Two-Way Hold-Out
    # -----------------------------------------------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(X_train_pool, y_train_pool, test_size=0.2, random_state=seed)

    # Initialize and fit a clean Decision Tree variant inside this technique block
    dt_model_3 = DecisionTreeClassifier(random_state=seed)
    dt_model_3.fit(X_train, y_train)
    record_metrics("3. Two-Way Hold-Out (80:20)", y_test_pool, dt_model_3.predict(X_test_pool), seed)


    # -----------------------------------------------------------------------------
    # 4. Three-Way Hold-Out
    # -----------------------------------------------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(X_train_pool, y_train_pool, test_size=0.25, random_state=seed)

    # Initialize and fit a clean Decision Tree variant inside this technique block
    dt_model_4 = DecisionTreeClassifier(random_state=seed)
    dt_model_4.fit(X_train, y_train)
    # Validation evaluates internal sub-split; Testing evaluates the independent 20% test set
    record_metrics("4. Three-Way Hold-Out (Validation Set)", y_val, dt_model_4.predict(X_val), seed)
    record_metrics("4. Three-Way Hold-Out (Testing Set)", y_test_pool, dt_model_4.predict(X_test_pool), seed)


    # -----------------------------------------------------------------------------
    # 5. Stratified Two-Way Hold-Out
    # -----------------------------------------------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(X_train_pool, y_train_pool, test_size=0.2, random_state=seed, stratify=y_train_pool)

    # Initialize and fit a clean Decision Tree variant inside this technique block
    dt_model_5 = DecisionTreeClassifier(random_state=seed)
    dt_model_5.fit(X_train, y_train)
    record_metrics("5. Stratified Two-Way Hold-Out", y_test_pool, dt_model_5.predict(X_test_pool), seed)


    # -----------------------------------------------------------------------------
    # 6. Stratified Three-Way Hold-Out
    # -----------------------------------------------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(X_train_pool, y_train_pool, test_size=0.25, random_state=seed, stratify=y_train_pool)

    # Initialize and fit a clean Decision Tree variant inside this technique block
    dt_model_6 = DecisionTreeClassifier(random_state=seed)
    dt_model_6.fit(X_train, y_train)
    record_metrics("6. Stratified Three-Way Hold-Out (Validation Set)", y_val, dt_model_6.predict(X_val), seed)
    record_metrics("6. Stratified Three-Way Hold-Out (Testing Set)", y_test_pool, dt_model_6.predict(X_test_pool), seed)


    # -----------------------------------------------------------------------------
    # 7. K-Fold Cross-Validation
    # -----------------------------------------------------------------------------
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    accs, precs, recs, f1s = [], [], [], []
    for train_fold_idx, _ in kf.split(X_train_pool, y_train_pool):
        X_tr_f = X_train_pool.iloc[train_fold_idx]
        y_tr_f = y_train_pool.iloc[train_fold_idx]

        # Initialize and fit a clean Decision Tree variant inside this validation fold split
        dt_model_7 = DecisionTreeClassifier(random_state=seed)
        dt_model_7.fit(X_tr_f, y_tr_f)

        preds = dt_model_7.predict(X_test_pool)
        accs.append(accuracy_score(y_test_pool, preds))
        precs.append(precision_score(y_test_pool, preds, zero_division=0))
        recs.append(recall_score(y_test_pool, preds, zero_division=0))
        f1s.append(f1_score(y_test_pool, preds, zero_division=0))

    all_records.append({
        'Seed': seed, 'Technique Splitting Design': "7. K-Fold Cross-Validation (k=5)",
        'Accuracy': np.mean(accs), 'Precision': np.mean(precs), 'Recall': np.mean(recs), 'F1-Score': np.mean(f1s)
    })


    # -----------------------------------------------------------------------------
    # 8. Group K-Fold Cross-Validation
    # -----------------------------------------------------------------------------
    gkf = GroupKFold(n_splits=5)
    accs, precs, recs, f1s = [], [], [], []
    for train_fold_idx, _ in gkf.split(X_train_pool, y_train_pool, groups=groups_train):
        X_tr_f = X_train_pool.iloc[train_fold_idx]
        y_tr_f = y_train_pool.iloc[train_fold_idx]

        # Initialize and fit a clean Decision Tree variant inside this validation fold split
        dt_model_8 = DecisionTreeClassifier(random_state=seed)
        dt_model_8.fit(X_tr_f, y_tr_f)

        preds = dt_model_8.predict(X_test_pool)
        accs.append(accuracy_score(y_test_pool, preds))
        precs.append(precision_score(y_test_pool, preds, zero_division=0))
        recs.append(recall_score(y_test_pool, preds, zero_division=0))
        f1s.append(f1_score(y_test_pool, preds, zero_division=0))

    all_records.append({
        'Seed': seed, 'Technique Splitting Design': "8. Group K-Fold Cross-Validation (k=5)",
        'Accuracy': np.mean(accs), 'Precision': np.mean(precs), 'Recall': np.mean(recs), 'F1-Score': np.mean(f1s)
    })


    # -----------------------------------------------------------------------------
    # 9. Stratified Group K-Fold Cross-Validation
    # -----------------------------------------------------------------------------
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    accs, precs, recs, f1s = [], [], [], []
    for train_fold_idx, _ in sgkf.split(X_train_pool, y_train_pool, groups=groups_train):
        X_tr_f = X_train_pool.iloc[train_fold_idx]
        y_tr_f = y_train_pool.iloc[train_fold_idx]

        # Initialize and fit a clean Decision Tree variant inside this validation fold split
        dt_model_9 = DecisionTreeClassifier(random_state=seed)
        dt_model_9.fit(X_tr_f, y_tr_f)

        preds = dt_model_9.predict(X_test_pool)
        accs.append(accuracy_score(y_test_pool, preds))
        precs.append(precision_score(y_test_pool, preds, zero_division=0))
        recs.append(recall_score(y_test_pool, preds, zero_division=0))
        f1s.append(f1_score(y_test_pool, preds, zero_division=0))

    all_records.append({
        'Seed': seed, 'Technique Splitting Design': "9. Stratified Group K-Fold CV (k=5)",
        'Accuracy': np.mean(accs), 'Precision': np.mean(precs), 'Recall': np.mean(recs), 'F1-Score': np.mean(f1s)
    })

# Convert list to structured data and aggregate the average performance over all seeds
master_df = pd.DataFrame(all_records)
summary_df = master_df.groupby('Technique Splitting Design').mean().reset_index().drop(columns=['Seed'])

# Display the summary table cleanly formatted
print("=== SYSTEM PERFORMANCE SUMMARY (AVERAGED OVER SEEDS 2-100) ===")
print(summary_df.to_string(index=False, formatters={
    'Accuracy': '{:,.4f}'.format,
    'Precision': '{:,.4f}'.format,
    'Recall': '{:,.4f}'.format,
    'F1-Score': '{:,.4f}'.format
}))

# -----------------------------------------------------------------------------
# Isolate the definitive optimal architecture based on aggregated seed run accuracy
# -----------------------------------------------------------------------------
best_technique_idx = summary_df['Accuracy'].idxmax()
best_technique = summary_df.loc[best_technique_idx]

print("\n" + "="*77)
print("★ BEST PERFORMING SPLITTING TECHNIQUE OVERALL ★")
print(f"Technique: {best_technique['Technique Splitting Design']}")
print(f"Accuracy : {best_technique['Accuracy']:.4f} (Highest Mean Test Score)")
print(f"Precision: {best_technique['Precision']:.4f}")
print(f"Recall   : {best_technique['Recall']:.4f}")
print(f"F1-Score : {best_technique['F1-Score']:.4f}")
print("="*77)

--- Processing Splitting Designs on Train/Test Split Pool (Seeds 2 to 100) ---

=== SYSTEM PERFORMANCE SUMMARY (AVERAGED OVER SEEDS 2-100) ===
                       Technique Splitting Design Accuracy Precision Recall F1-Score
                               1. Random Hold-Out   0.6974    0.5630 0.5710   0.5645
                             2. Shuffled Hold-Out   0.6974    0.5630 0.5710   0.5645
                      3. Two-Way Hold-Out (80:20)   0.6947    0.5611 0.5599   0.5575
              4. Three-Way Hold-Out (Testing Set)   0.6910    0.5541 0.5579   0.5532
           4. Three-Way Hold-Out (Validation Set)   0.6968    0.5579 0.5714   0.5622
                   5. Stratified Two-Way Hold-Out   0.6978    0.5631 0.5678   0.5629
   6. Stratified Three-Way Hold-Out (Testing Set)   0.6982    0.5642 0.5673   0.5632
6. Stratified Three-Way Hold-Out (Validation Set)   0.6967    0.5652 0.5793   0.5703
                 7. K-Fold Cross-Validation (k=5)   0.6991    0.5659 0.5685   0.5644
       

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, StratifiedGroupKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

# 1. Load the dataset
df = pd.read_csv('diabetes (1).csv')
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Synthesize a 'Group' parameter to demonstrate Group K-Folds
np.random.seed(42)
groups = np.random.randint(0, 100, size=len(df)) # 100 simulated patient IDs

# Initialize Model
dt_model = DecisionTreeClassifier(random_state=42)

# List to hold evaluation records across all values of k
summary_records = []
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1']

print("--- Processing Cross-Validation Loops (k = 2 to 20) ---\n")

# Loop through values of k from 2 to 20
for k in range(2, 21):
    print(f"Evaluating splits for k = {k}...")

    # -----------------------------------------------------------------------------
    # A. K-Fold Cross-Validation
    # -----------------------------------------------------------------------------
    kf = KFold(n_splits=k, shuffle=True, random_state=24)
    cv_kf = cross_validate(dt_model, X, y, cv=kf, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'K-Fold CV', 'k': k,
        'Accuracy': np.mean(cv_kf['test_accuracy']),
        'Precision': np.mean(cv_kf['test_precision']),
        'Recall': np.mean(cv_kf['test_recall']),
        'F1-Score': np.mean(cv_kf['test_f1'])
    })

    # -----------------------------------------------------------------------------
    # B. Group K-Fold Cross-Validation
    # -----------------------------------------------------------------------------
    gkf = GroupKFold(n_splits=k)
    cv_gkf = cross_validate(dt_model, X, y, groups=groups, cv=gkf, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'Group K-Fold CV', 'k': k,
        'Accuracy': np.mean(cv_gkf['test_accuracy']),
        'Precision': np.mean(cv_gkf['test_precision']),
        'Recall': np.mean(cv_gkf['test_recall']),
        'F1-Score': np.mean(cv_gkf['test_f1'])
    })

    # -----------------------------------------------------------------------------
    # C. Stratified K-Fold Cross-Validation
    # -----------------------------------------------------------------------------
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=24)
    cv_skf = cross_validate(dt_model, X, y, cv=skf, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'Stratified K-Fold CV', 'k': k,
        'Accuracy': np.mean(cv_skf['test_accuracy']),
        'Precision': np.mean(cv_skf['test_precision']),
        'Recall': np.mean(cv_skf['test_recall']),
        'F1-Score': np.mean(cv_skf['test_f1'])
    })

    # -----------------------------------------------------------------------------
    # D. Stratified Group K-Fold Cross-Validation
    # -----------------------------------------------------------------------------
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=24)
    cv_sgkf = cross_validate(dt_model, X, y, groups=groups, cv=sgkf, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'Stratified Group K-Fold CV', 'k': k,
        'Accuracy': np.mean(cv_sgkf['test_accuracy']),
        'Precision': np.mean(cv_sgkf['test_precision']),
        'Recall': np.mean(cv_sgkf['test_recall']),
        'F1-Score': np.mean(cv_sgkf['test_f1'])
    })

# Convert compiled results to a DataFrame
summary_df = pd.DataFrame(summary_records)

# -----------------------------------------------------------------------------
# FEATURE 1: Print all accuracies grouped neatly by k value
# -----------------------------------------------------------------------------
print("\n=== COMPLETE ACCURACY LOG BY FOLD COUNT (k) ===")
for k in range(2, 31):
    print(f"\n[ Folds: k = {k} ]")
    k_subset = summary_df[summary_df['k'] == k]
    for _, row in k_subset.iterrows():
        print(f"  -> {row['Technique']:<28} | Accuracy: {row['Accuracy']:.4f} | F1: {row['F1-Score']:.4f}")

# -----------------------------------------------------------------------------
# FEATURE 2: Find and isolate the absolute best configuration
# -----------------------------------------------------------------------------
best_model_idx = summary_df['Accuracy'].idxmax()
best_configuration = summary_df.loc[best_model_idx]

print("\n" + "="*77)
print("★ OPTIMAL MODEL SPLITTING DESIGN DETERMINATION ★")
print(f"Best Technique : {best_configuration['Technique']}")
print(f"Optimal Folds  : k = {best_configuration['k']}")
print(f"Highest Accuracy: {best_configuration['Accuracy']:.4f}")
print(f"Precision      : {best_configuration['Precision']:.4f}")
print(f"Recall         : {best_configuration['Recall']:.4f}")
print(f"F1-Score       : {best_configuration['F1-Score']:.4f}")
print("="*77)

--- Processing Cross-Validation Loops (k = 2 to 20) ---

Evaluating splits for k = 2...
Evaluating splits for k = 3...
Evaluating splits for k = 4...
Evaluating splits for k = 5...
Evaluating splits for k = 6...
Evaluating splits for k = 7...
Evaluating splits for k = 8...
Evaluating splits for k = 9...
Evaluating splits for k = 10...
Evaluating splits for k = 11...
Evaluating splits for k = 12...
Evaluating splits for k = 13...
Evaluating splits for k = 14...
Evaluating splits for k = 15...
Evaluating splits for k = 16...
Evaluating splits for k = 17...
Evaluating splits for k = 18...
Evaluating splits for k = 19...
Evaluating splits for k = 20...

=== COMPLETE ACCURACY LOG BY FOLD COUNT (k) ===

[ Folds: k = 2 ]
  -> K-Fold CV                    | Accuracy: 0.7057 | F1: 0.5893
  -> Group K-Fold CV              | Accuracy: 0.6966 | F1: 0.5540
  -> Stratified K-Fold CV         | Accuracy: 0.7266 | F1: 0.5908
  -> Stratified Group K-Fold CV   | Accuracy: 0.6760 | F1: 0.5255

[ Folds: k 

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, StratifiedGroupKFold, ShuffleSplit, cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load the dataset
df = pd.read_csv('diabetes (1).csv')
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Synthesize a 'Group' parameter to demonstrate Group K-Folds
np.random.seed(42)
groups = np.random.randint(0, 100, size=len(df)) # 100 simulated patient IDs

# Initialize Model
dt_model = DecisionTreeClassifier(random_state=42)

# List to hold evaluation records across all values of k and p
summary_records = []
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1']

print("--- Processing Splitting Loops & Resampling (Iterations 2 to 30) ---\n")

# Unified loop running from 2 to 30
# 'i' represents: Folds (k) for CV, Outbound Samples (p) for LpOCV, and Iterations (B) for Bootstrap
for i in range(2, 31):
    print(f"Evaluating systems for parameter value = {i}...")

    # -----------------------------------------------------------------------------
    # 1. K-Fold Cross-Validation (k = i)
    # -----------------------------------------------------------------------------
    kf = KFold(n_splits=i, shuffle=True, random_state=42)
    cv_kf = cross_validate(dt_model, X, y, cv=kf, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'K-Fold CV', 'Parameter Value': i,
        'Accuracy': np.mean(cv_kf['test_accuracy']),
        'Precision': np.mean(cv_kf['test_precision']),
        'Recall': np.mean(cv_kf['test_recall']),
        'F1-Score': np.mean(cv_kf['test_f1'])
    })

    # -----------------------------------------------------------------------------
    # 2. Group K-Fold Cross-Validation (k = i)
    # -----------------------------------------------------------------------------
    gkf = GroupKFold(n_splits=i)
    cv_gkf = cross_validate(dt_model, X, y, groups=groups, cv=gkf, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'Group K-Fold CV', 'Parameter Value': i,
        'Accuracy': np.mean(cv_gkf['test_accuracy']),
        'Precision': np.mean(cv_gkf['test_precision']),
        'Recall': np.mean(cv_gkf['test_recall']),
        'F1-Score': np.mean(cv_gkf['test_f1'])
    })

    # -----------------------------------------------------------------------------
    # 3. Stratified K-Fold Cross-Validation (k = i)
    # -----------------------------------------------------------------------------
    skf = StratifiedKFold(n_splits=i, shuffle=True, random_state=42)
    cv_skf = cross_validate(dt_model, X, y, cv=skf, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'Stratified K-Fold CV', 'Parameter Value': i,
        'Accuracy': np.mean(cv_skf['test_accuracy']),
        'Precision': np.mean(cv_skf['test_precision']),
        'Recall': np.mean(cv_skf['test_recall']),
        'F1-Score': np.mean(cv_skf['test_f1'])
    })

    # -----------------------------------------------------------------------------
    # 4. Stratified Group K-Fold Cross-Validation (k = i)
    # -----------------------------------------------------------------------------
    sgkf = StratifiedGroupKFold(n_splits=i, shuffle=True, random_state=42)
    cv_sgkf = cross_validate(dt_model, X, y, groups=groups, cv=sgkf, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'Stratified Group K-Fold CV', 'Parameter Value': i,
        'Accuracy': np.mean(cv_sgkf['test_accuracy']),
        'Precision': np.mean(cv_sgkf['test_precision']),
        'Recall': np.mean(cv_sgkf['test_recall']),
        'F1-Score': np.mean(cv_sgkf['test_f1'])
    })

    # -----------------------------------------------------------------------------
    # 5. Leave-p-Out Cross-Validation (p = i) [Stochastic Monte Carlo Approximation]
    # -----------------------------------------------------------------------------
    lpocv_approx = ShuffleSplit(n_splits=20, test_size=i, random_state=24)
    cv_lpo = cross_validate(dt_model, X, y, cv=lpocv_approx, scoring=scoring_metrics)
    summary_records.append({
        'Technique': 'Leave-p-Out CV (Approx)', 'Parameter Value': i,
        'Accuracy': np.mean(cv_lpo['test_accuracy']),
        'Precision': np.mean(cv_lpo['test_precision']),
        'Recall': np.mean(cv_lpo['test_recall']),
        'F1-Score': np.mean(cv_lpo['test_f1'])
    })

    # -----------------------------------------------------------------------------
    # 6. Bootstrap Sampling with Out-of-Bag (OOB) Evaluation (B = i Resamples)
    # -----------------------------------------------------------------------------
    boot_accs, boot_precs, boot_recs, boot_f1s = [], [], [], []

    for boot_run in range(i):
        # Sample N times with replacement to construct the Training split
        train_idx = np.random.choice(df.index, size=len(df), replace=True)
        # Unselected records automatically form the Out-of-Bag (OOB) Evaluation Set
        oob_idx = df.index[~df.index.isin(train_idx)]

        X_b_train, y_b_train = X.loc[train_idx], y.loc[train_idx]
        X_b_oob, y_b_oob = X.loc[oob_idx], y.loc[oob_idx]

        # Fit and predict using the OOB test set
        dt_model.fit(X_b_train, y_b_train)
        preds = dt_model.predict(X_b_oob)

        boot_accs.append(accuracy_score(y_b_oob, preds))
        boot_precs.append(precision_score(y_b_oob, preds, zero_division=0))
        boot_recs.append(recall_score(y_b_oob, preds, zero_division=0))
        boot_f1s.append(f1_score(y_b_oob, preds, zero_division=0))

    summary_records.append({
        'Technique': 'Bootstrap Sampling (OOB)', 'Parameter Value': i,
        'Accuracy': np.mean(boot_accs),
        'Precision': np.mean(boot_precs),
        'Recall': np.mean(boot_recs),
        'F1-Score': np.mean(boot_f1s)
    })

# Convert compiled results to a structured DataFrame
summary_df = pd.DataFrame(summary_records)

# -----------------------------------------------------------------------------
# FEATURE 1: Print all metric profiles grouped neatly by parameter value
# -----------------------------------------------------------------------------
print("\n=== COMPLETE METRIC PERFORMANCE LOG BY PARAMETER STEP ===")
for i in range(2, 31):
    print(f"\n[ Parameter Level (k / p / Loops) = {i} ]")
    step_subset = summary_df[summary_df['Parameter Value'] == i]
    for _, row in step_subset.iterrows():
        print(f"  -> {row['Technique']:<28} | Acc: {row['Accuracy']:.4f} | Prec: {row['Precision']:.4f} | Rec: {row['Recall']:.4f} | F1: {row['F1-Score']:.4f}")

# -----------------------------------------------------------------------------
# FEATURE 2: Find and isolate the absolute best technique and parameter setup
# -----------------------------------------------------------------------------
best_model_idx = summary_df['Accuracy'].idxmax()
best_configuration = summary_df.loc[best_model_idx]

print("\n" + "="*77)
print("★ OPTIMAL MODEL SPLITTING DESIGN DETERMINATION ★")
print(f"Best Technique   : {best_configuration['Technique']}")
print(f"Best Param Value : Parameter = {best_configuration['Parameter Value']}")
print(f"Highest Accuracy : {best_configuration['Accuracy']:.4f}")
print(f"Precision        : {best_configuration['Precision']:.4f}")
print(f"Recall           : {best_configuration['Recall']:.4f}")
print(f"F1-Score         : {best_configuration['F1-Score']:.4f}")
print("="*77)

--- Processing Splitting Loops & Resampling (Iterations 2 to 30) ---

Evaluating systems for parameter value = 2...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.p

Evaluating systems for parameter value = 3...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: 

Evaluating systems for parameter value = 4...


...

Evaluating systems for parameter value = 6...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Evaluating systems for parameter value = 7...


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Evaluating systems for parameter value = 8...
Evaluating systems for parameter value = 9...
Evaluating systems for parameter value = 10...
Evaluating systems for parameter value = 11...
Evaluating systems for parameter value = 12...
Evaluating systems for parameter value = 13...
Evaluating systems for parameter value = 14...
Evaluating systems for parameter value = 15...
Evaluating systems for parameter value = 16...
Evaluating systems for parameter value = 17...
Evaluating systems for parameter value = 18...
Evaluating systems for parameter value = 19...
Evaluating systems for parameter value = 20...
Evaluating systems for parameter value = 21...
Evaluating systems for parameter value = 22...
Evaluating systems for parameter value = 23...
Evaluating systems for parameter value = 24...
Evaluating systems for parameter value = 25...
Evaluating systems for parameter value = 26...
Evaluating systems for parameter value = 27...
Evaluating systems for parameter value = 28...
Evaluating syst

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import ShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load the dataset
df = pd.read_csv('diabetes (1).csv')
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Initialize Model
dt_model = DecisionTreeClassifier(random_state=42)

# List to hold evaluation records across all values of p
summary_records = []

print("--- Processing Leave-p-Out CV Manual Splitting Loops (p = 2 to 30) ---\n")

# Loop through validation hold-out sizes (p) from 2 to 30
for i in range(2, 31):
    # Setup Monte Carlo approximation with 25 independent splits
    lpocv_approx = ShuffleSplit(n_splits=25, test_size=i, random_state=100)

    # Temporary lists to track metrics across all 25 individual splits for the current p
    split_accuracies = []
    split_precisions = []
    split_recalls = []
    split_f1s = []

    # Manually iterate through each generated data split
    for train_idx, test_idx in lpocv_approx.split(X, y):
        # Partition the datasets into training and validation sets
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Train the Decision Tree on the current training split
        dt_model.fit(X_train, y_train)

        # Generate predictions on the hold-out validation split
        preds = dt_model.predict(X_test)

        # Calculate and record metrics for this explicit split
        split_accuracies.append(accuracy_score(y_test, preds))
        split_precisions.append(precision_score(y_test, preds, zero_division=0))
        split_recalls.append(recall_score(y_test, preds, zero_division=0))
        split_f1s.append(f1_score(y_test, preds, zero_division=0))

    # Isolate the highest accuracy scored among all splits for this step
    best_split_accuracy = np.max(split_accuracies)
    print(f"Evaluating system for parameter value p = {i:<2} | Highest Split Accuracy: {best_split_accuracy:.4f}")

    # Append the average cross-validated performance profile to the global summary records
    summary_records.append({
        'Technique': 'Leave-p-Out CV (Approx)', 'Parameter Value': i,
        'Accuracy': np.mean(split_accuracies),
        'Precision': np.mean(split_precisions),
        'Recall': np.mean(split_recalls),
        'F1-Score': np.mean(split_f1s)
    })

# Convert compiled results to a structured DataFrame
summary_df = pd.DataFrame(summary_records)

# -----------------------------------------------------------------------------
# FEATURE 1: Print all metric profiles grouped neatly by parameter value
# -----------------------------------------------------------------------------
print("\n=== COMPLETE METRIC PERFORMANCE LOG BY PARAMETER STEP ===")
for i in range(2, 31):
    print(f"\n[ Parameter Level (p) = {i} ]")
    step_subset = summary_df[summary_df['Parameter Value'] == i]
    for _, row in step_subset.iterrows():
        print(f"  -> {row['Technique']:<25} | Acc: {row['Accuracy']:.4f} | Prec: {row['Precision']:.4f} | Rec: {row['Recall']:.4f} | F1: {row['F1-Score']:.4f}")

# -----------------------------------------------------------------------------
# FEATURE 2: Find and isolate the absolute best technique and parameter setup
# -----------------------------------------------------------------------------
best_model_idx = summary_df['Accuracy'].idxmax()
best_configuration = summary_df.loc[best_model_idx]

print("\n" + "="*77)
print("★ OPTIMAL MODEL SPLITTING DESIGN DETERMINATION ★")
print(f"Best Technique   : {best_configuration['Technique']}")
print(f"Best Param Value : Parameter p = {best_configuration['Parameter Value']}")
print(f"Highest Accuracy : {best_configuration['Accuracy']:.4f} (Mean CV)")
print(f"Precision        : {best_configuration['Precision']:.4f}")
print(f"Recall           : {best_configuration['Recall']:.4f}")
print(f"F1-Score         : {best_configuration['F1-Score']:.4f}")
print("="*77)

--- Processing Leave-p-Out CV Manual Splitting Loops (p = 2 to 30) ---

Evaluating system for parameter value p = 2  | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 3  | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 4  | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 5  | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 6  | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 7  | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 8  | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 9  | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 10 | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 11 | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 12 | Highest Split Accuracy: 1.0000
Evaluating system for parameter value p = 13 | Highest Split Accuracy:

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import ShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Load the dataset
df = pd.read_csv('diabetes (1).csv')
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# List to hold evaluation records across the hyperparameter grid
summary_records = []

print("--- Running Multi-Loop Grid Search Optimization ---")
print("Evaluating: p (2-10) x Max Depth (1-10) x Seeds (20-100)\n")

# Loop 1: Validation hold-out sizes (p) from 2 to 10
for p in range(2, 11):
    print(f"Processing cross-validation grid for p = {p}...")

    # Loop 2: Decision Tree Maximum Depth from 1 to 10
    for depth in range(1, 11):

        # Loop 3: Random Seeds from 20 to 100 (Step of 10 for efficient runtime)
        for seed in range(20, 101, 10):

            # Setup Monte Carlo approximation with 25 independent splits using current seed
            lpocv_approx = ShuffleSplit(n_splits=25, test_size=p, random_state=seed)

            # Initialize Model with specific max_depth and match the random state
            dt_model = DecisionTreeClassifier(max_depth=depth, random_state=seed)

            # Arrays to track metrics across all 25 splits for this combination
            split_accuracies = []
            split_precisions = []
            split_recalls = []
            split_f1s = []

            # Manually iterate through each generated data split
            for train_idx, test_idx in lpocv_approx.split(X, y):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

                # Train the Decision Tree
                dt_model.fit(X_train, y_train)

                # Evaluate predictions
                preds = dt_model.predict(X_test)

                # Record metrics
                split_accuracies.append(accuracy_score(y_test, preds))
                split_precisions.append(precision_score(y_test, preds, zero_division=0))
                split_recalls.append(recall_score(y_test, preds, zero_division=0))
                split_f1s.append(f1_score(y_test, preds, zero_division=0))

            # Append the average cross-validated performance profile for this specific combination
            summary_records.append({
                'Technique': 'Leave-p-Out CV (Approx)',
                'p_value': p,
                'Max_Depth': depth,
                'Seed': seed,
                'Accuracy': np.mean(split_accuracies),
                'Precision': np.mean(split_precisions),
                'Recall': np.mean(split_recalls),
                'F1-Score': np.mean(split_f1s)
            })

# Convert compiled results to a structured DataFrame
summary_df = pd.DataFrame(summary_records)

# -----------------------------------------------------------------------------
# FEATURE 1: Print summarized performance profiles grouped by p-value
# -----------------------------------------------------------------------------
print("\n=== COMPACT METRIC PERFORMANCE LOG SUMMARY ===")
for p in range(2, 11):
    p_subset = summary_df[summary_df['p_value'] == p]
    # Find the best performing depth for this specific p-value to keep output readable
    best_at_p_idx = p_subset['Accuracy'].idxmax()
    best_row = p_subset.loc[best_at_p_idx]
    print(f"[ Hold-out Size p = {p:<2} ] Optimal Configuration found at -> Depth: {best_row['Max_Depth']:<2} | Seed: {best_row['Seed']:<3} | Peak CV Acc: {best_row['Accuracy']:.4f}")

# -----------------------------------------------------------------------------
# FEATURE 2: Find and isolate the absolute best configuration overall
# -----------------------------------------------------------------------------
best_model_idx = summary_df['Accuracy'].idxmax()
best_configuration = summary_df.loc[best_model_idx]

print("\n" + "="*77)
print("★ OPTIMAL MODEL SPLITTING & TREE DESIGN DETERMINATION ★")
print(f"Best Technique     : {best_configuration['Technique']}")
print(f"Optimal Param Value: Parameter p = {best_configuration['p_value']}")
print(f"Optimal Tree Depth : max_depth   = {best_configuration['Max_Depth']}")
print(f"Optimal Seed Value : random_state = {best_configuration['Seed']}")
print(f"Highest CV Accuracy: {best_configuration['Accuracy']:.4f}")
print(f"Precision          : {best_configuration['Precision']:.4f}")
print(f"Recall             : {best_configuration['Recall']:.4f}")
print(f"F1-Score           : {best_configuration['F1-Score']:.4f}")
print("="*77)

--- Running Multi-Loop Grid Search Optimization ---
Evaluating: p (2-10) x Max Depth (1-10) x Seeds (20-100)

Processing cross-validation grid for p = 2...
Processing cross-validation grid for p = 3...
Processing cross-validation grid for p = 4...
Processing cross-validation grid for p = 5...
Processing cross-validation grid for p = 6...
Processing cross-validation grid for p = 7...
Processing cross-validation grid for p = 8...
Processing cross-validation grid for p = 9...
Processing cross-validation grid for p = 10...

=== COMPACT METRIC PERFORMANCE LOG SUMMARY ===
[ Hold-out Size p = 2  ] Optimal Configuration found at -> Depth: 2  | Seed: 50  | Peak CV Acc: 0.8800
[ Hold-out Size p = 3  ] Optimal Configuration found at -> Depth: 5  | Seed: 70  | Peak CV Acc: 0.8533
[ Hold-out Size p = 4  ] Optimal Configuration found at -> Depth: 2  | Seed: 50  | Peak CV Acc: 0.8300
[ Hold-out Size p = 5  ] Optimal Configuration found at -> Depth: 5  | Seed: 100 | Peak CV Acc: 0.8480
[ Hold-out Size